In [1]:
import sys
sys.path.append('../')
import UTILS.utils as utils
import UTILS.utils_stats as stats
import pandas as pd

In [2]:
sentences = pd.read_csv(utils.ALL_SENTENCES_DF)

offensive_set = sentences[sentences["offensive"] == 1]
non_offensive_set = sentences[sentences["offensive"] == -1]
validation_set = pd.concat([offensive_set, non_offensive_set])

In [8]:
def get_results(df, column_name):
    TP = stats.get_true_positives(df, column_name, "offensive")
    print("True positives :", len(TP))
    FP = stats.get_false_positives(df, column_name, "offensive")
    print("False positives :", len(FP))
    FN = stats.get_false_negatives(df, column_name, "offensive")
    print("False negatives :", len(FN))
    TN = stats.get_true_negatives(df, column_name, "offensive")
    print("True negatives :", len(TN))


In [3]:
import google.generativeai as genai

genai.configure(api_key=utils.API_KEY())
model = genai.GenerativeModel('gemini-1.5-flash')

In [4]:
prompt_with_context = "The following text is a sentence from a book written in 1900 by an American travelling to China. Is the language racist ? Answer yes or no. Provide no other comments."

basic_prompt = "Is the language in the following text racist ? Answer yes or no. Provide no other comments."

def get_prompt(sentence, prompt):
    return prompt + "\n'" + sentence + "'\nAnswer :"

In [5]:
def prompt_model(sentence, prompt):
    response = model.generate_content(
    contents=[
        get_prompt(sentence, prompt)])
    return response.text

In [6]:
from tqdm import tqdm

def get_classification_from_gemini(prompt):
    gemini_response = []
    for sentence in tqdm(validation_set["text"]):
        response = prompt_model(sentence, prompt)
        gemini_response.append(response)
    return gemini_response

## CLASSIFICATION W/ CONTEXT

In [7]:
gemini_response = get_classification_from_gemini(prompt_with_context)

100%|██████████| 140/140 [00:51<00:00,  2.70it/s]


In [9]:
sentiment = []
for resp in gemini_response:
    if "yes" in resp.lower():
        sentiment.append(-1)
    elif "no" in resp.lower():
        sentiment.append(1)
    else:
        print(resp)

In [10]:
validation_set["sentiment"] = sentiment

In [11]:
stats.get_f1_score(validation_set, "sentiment", "offensive")

0.8533333333333333

In [12]:
get_results(validation_set, "sentiment")

True positives : 64
False positives : 16
False negatives : 6
True negatives : 54


## CLASSIFICATION WITHOUT CONTEXT

In [13]:
gemini_response_no_context = get_classification_from_gemini(basic_prompt)

100%|██████████| 140/140 [00:51<00:00,  2.71it/s]


In [14]:
sentiment = []
for resp in gemini_response_no_context:
    if "yes" in resp.lower():
        sentiment.append(-1)
    elif "no" in resp.lower():
        sentiment.append(1)
    else:
        print(resp)

validation_set["sentiment"] = sentiment

In [15]:
stats.get_f1_score(validation_set, "sentiment", "offensive")

0.8159999999999998

In [16]:
get_results(validation_set, "sentiment")

True positives : 51
False positives : 4
False negatives : 19
True negatives : 66
